# PBIT Feature Engineering

Extracts all 12 feature families from the long/tidy PBIT dataset and produces
`data/PBIT_features_r1.csv` — one row per experiment, ready for model training.

Feature name prefixes:

| Prefix | Family |
|--------|--------|
| `wc_` | Whole-cycle statistics |
| `ep_` | Endpoint / reversal geometry |
| `seg1_`…`seg8_` | Direction-aware segment descriptors |
| `sc_` | Segment contrast (outbound vs. return) |
| `d1_` | First-difference dynamics |
| `d2_` | Second-difference dynamics |
| `roll5_` `roll15_` | Rolling-statistic summaries |
| `ls_` | Local-shape descriptors |
| `tr_` | Trend descriptors |
| `fq_` | Frequency descriptors |
| `mq_` | Missingness / quality indicators |

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pbit.data_validation import load_long_dataset
from pbit.features import extract_features, feature_names

DATA_PATH    = PROJECT_ROOT / "data" / "PBIT_atuador_DESCARAC_r1_long_clean.csv"
OUTPUT_PATH  = PROJECT_ROOT / "data" / "PBIT_features_r1.csv"
PROJECT_ROOT

## Load Dataset

In [ ]:
df = load_long_dataset(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Experiments: {df['Experiment_Index'].nunique():,}")
df.head(5)

## Inspect Feature Catalogue

Check the full list of feature names and their count before running extraction.

In [ ]:
names = feature_names()
print(f"Total features: {len(names)}")
print()

import re
from collections import Counter

prefixes = [re.match(r'^([a-z0-9]+)_', n).group(1) for n in names]
prefix_counts = Counter(prefixes)
for prefix, count in sorted(prefix_counts.items(), key=lambda x: -x[1]):
    print(f"  {prefix:>8}_  {count:>3} features")

## Extract Features

This processes all 1,490 experiments. Expect ~30–90 seconds depending on hardware.

In [ ]:
import time

t0 = time.time()
features_df = extract_features(df)
elapsed = time.time() - t0

print(f"Done in {elapsed:.1f}s")
print(f"Feature matrix: {features_df.shape[0]:,} rows × {features_df.shape[1]} columns")
features_df.head(3)

## Metadata vs. Feature Columns

In [ ]:
META_COLS = ["Experiment_Index", "Serial_Number", "Serial_Run_Index",
             "Fault_Mode", "Severity_Index"]
feat_cols = [c for c in features_df.columns if c not in META_COLS]

print(f"Metadata columns : {len(META_COLS)}")
print(f"Feature columns  : {len(feat_cols)}")
print(f"Total columns    : {features_df.shape[1]}")

## Missing-Value Audit

Expected: only experiments with known bad measurements (11 runs) should produce
any NaNs, and only in features that depend on the signal values of those samples.

In [ ]:
import pandas as pd

nan_per_col = features_df[feat_cols].isna().sum()
nan_cols = nan_per_col[nan_per_col > 0].sort_values(ascending=False)

print(f"Feature columns with any NaN: {len(nan_cols)} / {len(feat_cols)}")
if len(nan_cols) > 0:
    print()
    print(nan_cols.to_string())

In [ ]:
# Rows with any NaN feature
nan_rows = features_df[features_df[feat_cols].isna().any(axis=1)][
    META_COLS + ["mq_missing_count"]
].copy()
print(f"Experiments with at least one NaN feature: {len(nan_rows)}")
nan_rows

## Feature Statistics by Fault Mode

Quick sanity check: do any whole-cycle features separate fault modes visually?

In [ ]:
import matplotlib.pyplot as plt

check_features = ["wc_mean", "wc_std", "wc_range", "wc_rms",
                  "d1_mean_abs_slope", "tr_slope", "fq_spectral_entropy"]

fault_order = ["N01", "N02", "F01", "F02", "F03", "F04", "F05", "F06"]

fig, axes = plt.subplots(1, len(check_features), figsize=(18, 4), sharey=False)
for ax, feat in zip(axes, check_features):
    data = [features_df.loc[features_df["Fault_Mode"] == fm, feat].dropna().values
            for fm in fault_order]
    ax.boxplot(data, tick_labels=fault_order, vert=True)
    ax.set_title(feat, fontsize=8)
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)

fig.suptitle("Feature distributions by fault mode", fontsize=10)
plt.tight_layout()
plt.show()

## F06 vs. N01 Feature Comparison

F06 is the critical minority class. Check whether any features clearly separate it from N01.

In [ ]:
f06_df  = features_df[features_df["Fault_Mode"] == "F06"][feat_cols]
n01_df  = features_df[features_df["Fault_Mode"] == "N01"][feat_cols]

comparison = pd.DataFrame({
    "F06_mean":  f06_df.mean(),
    "N01_mean":  n01_df.mean(),
    "F06_std":   f06_df.std(),
    "N01_std":   n01_df.std(),
})
comparison["abs_diff"] = (comparison["F06_mean"] - comparison["N01_mean"]).abs()
comparison["ratio"] = comparison["abs_diff"] / (comparison["N01_std"] + 1e-12)

print("Top 20 features separating F06 from N01 (by standardised difference):")
comparison.sort_values("ratio", ascending=False).head(20)

## Segment Feature Heatmap

Mean value per segment per fault mode — shows which stroke region differs across classes.

In [ ]:
import numpy as np

seg_mean_cols = [f"seg{i}_mean" for i in range(1, 9)]
heatmap_data = (
    features_df.groupby("Fault_Mode")[seg_mean_cols]
    .mean()
    .reindex(fault_order)
)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="RdBu_r")
ax.set_xticks(range(8))
ax.set_xticklabels([f"seg{i}" for i in range(1, 9)])
ax.set_yticks(range(len(fault_order)))
ax.set_yticklabels(fault_order)
ax.set_xlabel("Segment (1=outbound start … 8=return end)")
ax.set_title("Mean segment value by fault mode")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Save Feature Matrix

In [ ]:
features_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {features_df.shape[0]:,} rows × {features_df.shape[1]} columns")
print(f"  Metadata : {len(META_COLS)}")
print(f"  Features : {len(feat_cols)}")